In [7]:
import zipfile
import os
import warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore")
import pandas as pd
from huggingface_hub import hf_hub_download
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel

# 1. Download the raw zip from the repo (this file is on main)
zip_path = hf_hub_download(
    repo_id="takala/financial_phrasebank",
    repo_type="dataset",
    filename="data/FinancialPhraseBank-v1.0.zip",
)

# 2. Find and read the "all agree" file inside the zip
with zipfile.ZipFile(zip_path) as z:
    name = next(n for n in z.namelist() if n.lower().endswith("sentences_allagree.txt"))
    raw = z.read(name).decode("iso-8859-1")

# 3. Parse "sentence@label" lines (split on the LAST "@", since sentences can contain "@")
label_names = ["negative", "neutral", "positive"]
label2id = {n: i for i, n in enumerate(label_names)}

rows = []
for line in raw.splitlines():
    line = line.strip()
    if not line:
        continue
    sentence, label = line.rsplit("@", 1)
    rows.append({"sentence": sentence, "label": label2id[label.strip()]})

# 4. Build a Hugging Face Dataset with the same schema as the original
features = Features({
    "sentence": Value("string"),
    "label": ClassLabel(names=label_names),
})
ds = DatasetDict({"train": Dataset.from_pandas(pd.DataFrame(rows), features=features)})

print(ds)
print(ds["train"][0])
print(len(ds["train"]))  # should be 2264

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 2264
    })
})
{'sentence': 'According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .', 'label': 1}
2264
